In [1]:
import torch
import numpy as np
from utilities import (
    KolmogorovDataset,
    FNOGenerator,
    FNODiscriminator,
    NavierStokesLoss,
    WGAFNOGPTrainer,
    Rollout,
)
from torch.utils.data import DataLoader, random_split

In [2]:
!pip list

Package                   Version
------------------------- ----------------
aiohappyeyeballs          2.6.1
aiohttp                   3.13.3
aiosignal                 1.4.0
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asciitree                 0.3.3
asttokens                 3.0.1
async-lru                 2.2.0
async-timeout             5.0.1
attrs                     25.4.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
bokeh                     3.9.0
certifi                   2026.2.25
cffi                      2.0.0
cftime                    1.6.5
charset-normalizer        3.4.6
click                     8.3.1
comm                      0.2.3
configmypy                0.2.0
contourpy                 1.3.2
cycler                    0.12.1
debugpy                   1.8.20
decorator     

In [3]:
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../datasets/Snapshots/snapshots_64x64_use.npy"
H, W      = 64, 64
SEQ_LEN   = 5
EPOCHS    = 2

# ── Hiperparámetros ajustados para GPU ≤8GB ────────────────
BATCH     = 8
MODES     = 8
HIDDEN    = 32
N_LAYERS  = 3
N_CRITIC  = 3

# ── Estocasticidad ─────────────────────────────────────────
# z_dim canales de ruido N(0,I) concatenados a wₙ en cada paso.
# z_dim=0 recupera el generador determinista original.
# z_dim=4 es un buen punto de partida: suficiente variabilidad
# sin dominar la señal de vorticidad (1 canal).
Z_DIM     = 4

# ── Física ─────────────────────────────────────────────────
NU        = 1e-3
DT        = 0.1

# ── Pérdidas ───────────────────────────────────────────────
LAMBDA_GP = 10.0
ALPHA_MSE = 1.0
BETA_PHYS = 0.1

# ── Dataset ────────────────────────────────────────────────
dataset   = KolmogorovDataset(DATA_PATH, seq_len=SEQ_LEN)
train_len = int(0.8 * len(dataset))
val_len   = len(dataset) - train_len
train_dataset, val_dataset = random_split(dataset, [train_len, val_len])

train_loader = DataLoader(
    train_dataset, batch_size=BATCH, shuffle=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH, shuffle=False, num_workers=4,
)

# ── Modelos ────────────────────────────────────────────────
G = FNOGenerator(
    hidden_ch=HIDDEN, modes1=MODES, modes2=MODES,
    n_layers=N_LAYERS, z_dim=Z_DIM,
)
D       = FNODiscriminator(seq_len=SEQ_LEN+1, hidden_ch=HIDDEN, modes1=MODES, modes2=MODES, n_layers=N_LAYERS)
ns_loss = NavierStokesLoss(H, W, nu=NU, dt=DT, device=DEVICE)

# ── Trainer ────────────────────────────────────────────────
trainer = WGAFNOGPTrainer(
    generator=G,
    discriminator=D,
    ns_loss=ns_loss,
    device=DEVICE,
    lr_G=1e-4,
    lr_D=1e-4,
    n_critic=N_CRITIC,
    lambda_gp=LAMBDA_GP,
    alpha_mse=ALPHA_MSE,
    beta_phys=BETA_PHYS,
)

# Opcional: reanudar desde checkpoint
# start_epoch = trainer.load_checkpoint("best_checkpoint_wgafnogp.pt")

history = trainer.fit(train_loader, val_loader, n_epochs=EPOCHS, log_every=10)

# ── Rollout ────────────────────────────────────────────────
sample = next(iter(val_loader))
w0     = sample[0][:, 0]   # (B, 1, H, W) condición inicial
w_true = sample[2]         # (B, seq_len+1, H, W) ground truth

roller = Rollout(G, DEVICE)

# Rollout estocástico — z fresco en cada paso (por defecto)
traj_stochastic = roller.run(w0, n_steps=SEQ_LEN, w_true=w_true)

# Rollout con z fijo — trayectoria reproducible para figuras del paper
z_fixed = torch.randn(w0.shape[0], Z_DIM, H, W)
traj_fixed = roller.run(w0, n_steps=SEQ_LEN, w_true=w_true, z=z_fixed)

print("\nRollout completado. Métricas (estocástico):")
print("Energía:       ", roller.metrics["energy"])
print("RMSE:          ", roller.metrics["rmse"])
print("Error relativo:", roller.metrics["rel_error"])

Dataset cargado: 1152 trayectorias × 95 ventanas = 109,440 muestras  |  seq_len=5  |  H×W=64×64


/home/joseluis/miniconda3/envs/cfd-project-personal/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass

KeyboardInterrupt



In [2]:
"""
WGAFNOGP v2 — Rollout Autoregresivo, Flujo de Kolmogorov 2D
=============================================================
Dataset:   .npy shape (N, T, H, W)  — solo vorticidad
           e.g.  (1152, 100, 64, 64)

Generador: wₙ (1 canal) → wₙ₊₁  via FNO + residual connection
Discriminador: trayectoria (B, T+1, H, W) → escalar (WGAN)

Pérdidas:
  L_D = 𝔼[D(fake)] − 𝔼[D(real)] + λ·GP
  L_G = −𝔼[D(fake)] + α·MSE(pred, vort_{n+1}) + β·L_NS

  L_NS = ‖∂w/∂t + u·∂w/∂x + v·∂w/∂y − ν·∇²w‖²
         calculado en Fourier (exacto en malla periódica)
         u, v recuperados de w via función de corriente

Estructura:
  1. SpectralConv2d     — convolución en espacio de Fourier
  2. FNOBlock           — capa FNO completa
  3. FNOGenerator       — G_θ: wₙ → wₙ₊₁
  4. FNODiscriminator   — D_φ: trayectoria → escalar
  5. KolmogorovDataset  — carga .npy (N,T,H,W), ventanas deslizantes
  6. NavierStokesLoss   — residuo NS puro en Fourier (sin IBM)
  7. GradientPenalty    — GP en norma L² funcional sobre trayectorias (clase)
  8. Rollout            — generación autoregresiva con métricas
  9. WGAFNOGPTrainer    — loop adversarial completo
 10. main              — ejemplo de uso

Nuevas mejoras (v3):
  - [MEJORA 1] Verbosidad con tqdm y logging a archivo.
  - [MEJORA 2] Métricas adicionales: energía, enstrofía, espectro de energía.
  - [MEJORA 3] Visualización periódica de campos de vorticidad y curvas de pérdida.
  - [MEJORA 4] Scheduler de LR con ReduceLROnPlateau para G y D.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
from pathlib import Path
import logging
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# Configuración de logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler("training.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════
# 1. DATASET
# ═══════════════════════════════════════════════════════════
class KolmogorovDataset(Dataset):
    """
    Carga vorticidad Kolmogorov desde .npy shape (N, T, H, W).
    Construye ventanas sobre la marcha sin duplicar datos.
    """

    def __init__(self, path, seq_len=10):
        path = Path(path)
        assert path.exists(), f"No se encontró: {path}"

        w = np.load(path)
        if w.ndim == 3:
            w = w[None]
        assert w.ndim == 4, f"Esperado (N, T, H, W), got {w.shape}"
        w = w.astype(np.float32)

        N, T, H, W = w.shape
        assert T > seq_len, f"T={T} debe ser > seq_len={seq_len}"
        self.seq_len = seq_len
        self.H, self.W = H, W
        self.N, self.T = N, T

        self.w_mean = float(w.mean())
        # [FIX 9] warning explícito si el dataset es casi constante
        raw_std = float(w.std())
        if raw_std < 1e-6:
            import warnings
            warnings.warn(
                f"w_std={raw_std:.2e} — dataset casi constante, "
                "normalización degenerada. Verifica el archivo .npy."
            )
        self.w_std = raw_std + 1e-8
        w = (w - self.w_mean) / self.w_std

        self.w = w                      # shape (N, T, H, W)
        self.n_windows = T - seq_len

        logger.info(
            f"Dataset cargado: {N} trayectorias × {self.n_windows} ventanas "
            f"= {len(self):,} muestras  |  seq_len={seq_len}  |  H×W={H}×{W}"
        )

    def __len__(self):
        return self.N * self.n_windows

    def __getitem__(self, idx):
        """
        Retorna:
          seq_in:  (seq_len, 1, H, W)
          seq_out: (seq_len, 1, H, W)
          traj:    (seq_len+1, H, W)
        """
        traj_idx = idx // self.n_windows
        t0       = idx % self.n_windows

        traj = self.w[traj_idx, t0 : t0 + self.seq_len + 1]  # (seq_len+1, H, W)
        # [FIX 9] .copy() obligatorio: el slice en dim T produce un array
        # no contiguo en memoria; torch.from_numpy requiere array contiguo
        # y sin .copy() los workers del DataLoader pueden producir race conditions.
        traj_tensor = torch.from_numpy(traj.copy())
        seq_in      = traj_tensor[:-1].unsqueeze(1)   # (seq_len, 1, H, W)
        seq_out     = traj_tensor[1:].unsqueeze(1)    # (seq_len, 1, H, W)

        return seq_in, seq_out, traj_tensor


# ═══════════════════════════════════════════════════════════
# 2. BLOQUE ESPECTRAL (sin cambios)
# ═══════════════════════════════════════════════════════════
class SpectralConv2d(nn.Module):
    """
    Convolución integral en espacio de Fourier.
    v' = F⁻¹( R · F(v) )
    R ∈ ℂ^{in_ch × out_ch × modes1 × modes2}  — parámetros aprendidos
    """

    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1, self.modes2 = modes1, modes2
        scale = 1.0 / (in_ch * out_ch)
        self.W1 = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))
        self.W2 = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))

    def _mul(self, x, w):
        return torch.einsum("bixy,ioxy->boxy", x, w)

    def forward(self, x):
        B, C, H, W = x.shape
        xf  = torch.fft.rfft2(x)
        out = torch.zeros(B, self.W1.shape[1], H, W // 2 + 1,
                          dtype=torch.cfloat, device=x.device)
        out[:, :,  :self.modes1, :self.modes2] = self._mul(xf[:, :,  :self.modes1, :self.modes2], self.W1)
        out[:, :, -self.modes1:, :self.modes2] = self._mul(xf[:, :, -self.modes1:, :self.modes2], self.W2)
        return torch.fft.irfft2(out, s=(H, W))


# ═══════════════════════════════════════════════════════════
# 3. CAPA FNO (sin cambios)
# ═══════════════════════════════════════════════════════════
class FNOBlock(nn.Module):
    """
    v' = σ( F⁻¹(R·F(v)) + W·v )
    Término integral (global) + transformación local punto a punto.
    """

    def __init__(self, ch, modes1, modes2):
        super().__init__()
        self.spectral = SpectralConv2d(ch, ch, modes1, modes2)
        self.local    = nn.Conv2d(ch, ch, kernel_size=1)
        self.norm     = nn.InstanceNorm2d(ch)

    def forward(self, x):
        return F.gelu(self.norm(self.spectral(x) + self.local(x)))


# ═══════════════════════════════════════════════════════════
# 4. GENERADOR (sin cambios)
# ═══════════════════════════════════════════════════════════
class FNOGenerator(nn.Module):
    """
    G_θ: (wₙ, z) → wₙ₊₁   — generador estocástico

    Input:  (B, 1, H, W)   — vorticidad en t=n
            z ~ N(0,I)      — vector de ruido latente (B, z_dim, H, W)
                              broadcast espacial: el mismo z se repite en H×W
    Output: (B, 1, H, W)   — vorticidad en t=n+1
    """

    def __init__(self, hidden_ch=64, modes1=12, modes2=12, n_layers=4, z_dim=4):
        super().__init__()
        self.z_dim = z_dim
        self.lift = nn.Conv2d(1 + z_dim, hidden_ch, kernel_size=1)
        self.layers = nn.ModuleList([
            FNOBlock(hidden_ch, modes1, modes2) for _ in range(n_layers)
        ])
        self.proj = nn.Sequential(
            nn.Conv2d(hidden_ch, hidden_ch // 2, kernel_size=1),
            nn.GELU(),
            nn.Conv2d(hidden_ch // 2, 1, kernel_size=1),
        )

    def forward(self, w_n, z=None):
        B, _, H, W = w_n.shape
        if self.z_dim > 0:
            if z is None:
                z = torch.randn(B, self.z_dim, H, W, device=w_n.device)
            h = self.lift(torch.cat([w_n, z], dim=1))
        else:
            h = self.lift(w_n)
        for layer in self.layers:
            h = layer(h)
        return w_n + self.proj(h)


# ═══════════════════════════════════════════════════════════
# 5. DISCRIMINADOR (sin cambios)
# ═══════════════════════════════════════════════════════════
class FNODiscriminator(nn.Module):
    def __init__(self, seq_len, hidden_ch=64, modes1=12, modes2=12, n_layers=4):
        super().__init__()
        self.seq_len = seq_len
        self.temporal_mix = nn.Sequential(
            nn.Conv1d(seq_len, hidden_ch // 2, kernel_size=1),
            nn.GELU(),
            nn.Conv1d(hidden_ch // 2, hidden_ch, kernel_size=1),
        )
        self.layers = nn.ModuleList([
            FNOBlock(hidden_ch, modes1, modes2) for _ in range(n_layers)
        ])
        self.head = nn.Sequential(
            nn.Linear(hidden_ch, hidden_ch // 2),
            nn.GELU(),
            nn.Linear(hidden_ch // 2, 1),
        )

    def forward(self, traj):
        B, T, H, W = traj.shape
        x = traj.permute(0, 2, 3, 1).contiguous().reshape(B * H * W, T, 1)
        x = self.temporal_mix(x).squeeze(-1)
        x = x.reshape(B, H, W, -1).permute(0, 3, 1, 2)
        for layer in self.layers:
            x = layer(x)
        x = x.mean(dim=(-2, -1))
        return self.head(x)


# ═══════════════════════════════════════════════════════════
# 6. PÉRDIDA NS PURA EN FOURIER (sin cambios)
# ═══════════════════════════════════════════════════════════
class NavierStokesLoss(nn.Module):
    def __init__(self, H, W, nu=1e-3, dt=0.01, device="cpu"):
        super().__init__()
        self.nu = nu
        self.dt = dt

        kx = torch.fft.fftfreq(W, d=1.0 / W)
        ky = torch.fft.fftfreq(H, d=1.0 / H)
        KY, KX = torch.meshgrid(ky, kx, indexing="ij")
        K2 = KX ** 2 + KY ** 2
        K2_inv = K2.clone()
        K2_inv[0, 0] = 1.0

        self.register_buffer("KX", KX)
        self.register_buffer("KY", KY)
        self.register_buffer("K2", K2)
        self.register_buffer("K2_inv", K2_inv)

    def _velocity_from_vorticity(self, w):
        wf = torch.fft.fft2(w)
        wf[:, 0, 0] = 0
        psi = wf / self.K2_inv
        psi[:, 0, 0] = 0
        u = torch.fft.ifft2(1j * self.KY * psi).real
        v = torch.fft.ifft2(-1j * self.KX * psi).real
        return u, v

    def forward(self, w_n, w_pred):
        if torch.isnan(w_pred).any() or torch.isinf(w_pred).any():
            raise RuntimeError("NavierStokesLoss: w_pred contiene NaN/Inf")
        w_n = w_n.squeeze(1)
        w_pred = w_pred.squeeze(1)

        dwdt = (w_pred - w_n) / self.dt
        u, v = self._velocity_from_vorticity(w_n)

        wn_f = torch.fft.fft2(w_n)
        dwdx = torch.fft.ifft2(1j * self.KX * wn_f).real
        dwdy = torch.fft.ifft2(1j * self.KY * wn_f).real

        adv = u * dwdx + v * dwdy
        lap_w = torch.fft.ifft2(-self.K2 * wn_f).real

        residual = dwdt + adv - self.nu * lap_w
        return (residual ** 2).mean()


# ═══════════════════════════════════════════════════════════
# 7. GRADIENT PENALTY (sin cambios)
# ═══════════════════════════════════════════════════════════
class GradientPenalty(nn.Module):
    def __init__(self, lambda_gp=10.0):
        super().__init__()
        self.lambda_gp = lambda_gp

    def forward(self, discriminator, real_traj, fake_traj):
        assert real_traj.shape == fake_traj.shape
        B = real_traj.size(0)
        alpha = torch.rand(B, 1, 1, 1, device=real_traj.device)
        interp = (alpha * real_traj + (1 - alpha) * fake_traj).requires_grad_(True)

        score = discriminator(interp)
        grad = torch.autograd.grad(
            outputs=score,
            inputs=interp,
            grad_outputs=torch.ones_like(score),
            create_graph=True,
            retain_graph=True,
        )[0]
        grad_norm = grad.flatten(1).norm(2, dim=1)
        return self.lambda_gp * ((grad_norm - 1) ** 2).mean()


# ═══════════════════════════════════════════════════════════
# 8. ROLLOUT (sin cambios)
# ═══════════════════════════════════════════════════════════
class Rollout:
    def __init__(self, generator, device):
        self.generator = generator.to(device)
        self.device = device
        self.metrics = {}

    @torch.no_grad()
    def run(self, w0, n_steps, w_true=None, z=None):
        self.generator.eval()
        w_cur = w0.to(self.device)
        if z is not None:
            z = z.to(self.device)
        traj = [w_cur.squeeze(1).cpu()]
        for _ in range(n_steps):
            w_next = self.generator(w_cur, z=z)
            traj.append(w_next.squeeze(1).cpu())
            w_cur = w_next
        traj = torch.stack(traj, dim=1)
        self._compute_metrics(traj, w_true)
        self.generator.train()
        return traj

    def _compute_metrics(self, traj, w_true):
        energy = 0.5 * (traj ** 2).mean(dim=(0, 2, 3))
        self.metrics["energy"] = energy.numpy()
        if w_true is not None:
            T = min(traj.shape[1], w_true.shape[1])
            diff = traj[:, :T] - w_true[:, :T]
            self.metrics["rmse"] = diff.pow(2).mean(dim=(0, 2, 3)).sqrt().numpy()
            self.metrics["rel_error"] = (
                diff.norm(dim=(2, 3)) / (w_true[:, :T].norm(dim=(2, 3)) + 1e-8)
            ).mean(0).numpy()


# ═══════════════════════════════════════════════════════════
# 9. TRAINER ADVERSARIAL (MEJORADO)
# ═══════════════════════════════════════════════════════════
class WGAFNOGPTrainer:
    def __init__(
        self,
        generator,
        discriminator,
        ns_loss,
        device,
        lr_G=1e-4,
        lr_D=1e-4,
        n_critic=5,
        lambda_gp=10.0,
        alpha_mse=1.0,
        beta_phys=0.1,
        use_scheduler=True,           # Nuevo: activar scheduler
        scheduler_patience=5,         # Nuevo: paciencia para ReduceLROnPlateau
        scheduler_factor=0.5,         # Nuevo: factor de reducción
        log_dir="logs",               # Nuevo: directorio para guardar logs/plots
        vis_freq=10,                  # Nuevo: frecuencia de visualización (en épocas)
    ):
        self.G = generator.to(device)
        self.D = discriminator.to(device)
        self.ns_loss = ns_loss.to(device)
        self.device = device
        self.n_critic = n_critic
        self.lambda_gp = lambda_gp
        self.alpha_mse = alpha_mse
        self.beta_phys = beta_phys

        self.gp = GradientPenalty(lambda_gp=lambda_gp).to(device)

        self.opt_G = torch.optim.Adam(generator.parameters(), lr=lr_G, betas=(0.0, 0.9))
        self.opt_D = torch.optim.Adam(discriminator.parameters(), lr=lr_D, betas=(0.0, 0.9))

        # Schedulers opcionales
        self.use_scheduler = use_scheduler
        if use_scheduler:
            self.sched_G = torch.optim.lr_scheduler.ReduceLROnPlateau(
                self.opt_G, mode='min', factor=scheduler_factor, patience=scheduler_patience,
            )
            self.sched_D = torch.optim.lr_scheduler.ReduceLROnPlateau(
                self.opt_D, mode='min', factor=scheduler_factor, patience=scheduler_patience,
            )
        else:
            self.sched_G = self.sched_D = None

        self._critic_iter = None
        self.history = {
            "loss_D": [], "loss_G": [], "w_dist": [],
            "loss_mse": [], "loss_phys": [], "val_mse": [],
            "lr_G": [], "lr_D": []
        }

        # Para visualización
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.vis_freq = vis_freq

        # Datos de ejemplo para visualización (se tomarán del val_loader en el primer fit)
        self.val_example = None

    def _generate_sequence(self, seq_in, z=None):
        B, seq_len, _, H, W = seq_in.shape
        w_cur = seq_in[:, 0]
        fake = [w_cur.squeeze(1)]
        w_preds = []
        for t in range(seq_len):
            w_next = self.G(w_cur, z=z)
            fake.append(w_next.squeeze(1))
            w_preds.append(w_next)
            w_cur = w_next
        fake_traj = torch.stack(fake, dim=1)
        return fake_traj, w_preds

    def _physics_loss(self, seq_in, w_preds):
        total = 0.0
        for t, w_pred in enumerate(w_preds):
            w_n = seq_in[:, t]
            total = total + self.ns_loss(w_n, w_pred)
        return total / len(w_preds)

    def _step_D(self, loader):
        loss_list = []
        score_real_last = score_fake_last = None
        for _ in range(self.n_critic):
            try:
                seq_in, _, real_traj = next(self._critic_iter)
            except StopIteration:
                self._critic_iter = iter(loader)
                seq_in, _, real_traj = next(self._critic_iter)
            seq_in = seq_in.to(self.device)
            real_traj = real_traj.to(self.device)
            with torch.no_grad():
                fake_traj, _ = self._generate_sequence(seq_in)
            score_real = self.D(real_traj)
            score_fake = self.D(fake_traj)
            gp = self.gp(self.D, real_traj, fake_traj)
            loss_D = score_fake.mean() - score_real.mean() + gp
            self.opt_D.zero_grad()
            loss_D.backward()
            self.opt_D.step()
            loss_list.append(loss_D.item())
            score_real_last = score_real
            score_fake_last = score_fake
        w_dist = (score_real_last.mean() - score_fake_last.mean()).item()
        return float(np.mean(loss_list)), w_dist

    def _step_G(self, seq_in, seq_out):
        B, seq_len, _, H, W = seq_in.shape
        fake_traj, w_preds = self._generate_sequence(seq_in)
        loss_adv = -self.D(fake_traj).mean()
        loss_mse = sum(F.mse_loss(wp, gt) for wp, gt in zip(w_preds, seq_out.unbind(1))) / seq_len
        loss_phys = self._physics_loss(seq_in, w_preds)
        loss_G = loss_adv + self.alpha_mse * loss_mse + self.beta_phys * loss_phys
        self.opt_G.zero_grad()
        loss_G.backward()
        nn.utils.clip_grad_norm_(self.G.parameters(), max_norm=1.0)
        self.opt_G.step()
        return loss_G.item(), loss_mse.item(), loss_phys.item()

    @torch.no_grad()
    def _validate(self, loader):
        self.G.eval()
        total_mse = 0.0
        total_energy = 0.0
        total_enstrophy = 0.0
        n_batches = 0
        for seq_in, seq_out, _ in loader:
            seq_in = seq_in.to(self.device)
            seq_out = seq_out.to(self.device)
            _, w_preds = self._generate_sequence(seq_in)
            seq_len = seq_in.size(1)
            # MSE
            batch_mse = sum(F.mse_loss(wp, gt) for wp, gt in zip(w_preds, seq_out.unbind(1))) / seq_len
            total_mse += batch_mse.item()
            # Métricas físicas: energía y enstrofía en el último paso (por ejemplo)
            w_last = w_preds[-1].squeeze(1)  # (B, H, W)
            energy = 0.5 * (w_last ** 2).mean(dim=(1,2)).mean().item()
            total_energy += energy
            # Enstrofía: ½∫ (∇w)² dx = ½∫ |k|² |ŵ|² dk
            w_f = torch.fft.fft2(w_last)
            k2 = self.ns_loss.K2
            enstrophy = 0.5 * (k2 * w_f.abs()**2).mean().item()
            total_enstrophy += enstrophy
            n_batches += 1
        self.G.train()
        return total_mse / n_batches, total_energy / n_batches, total_enstrophy / n_batches

    def visualize(self, epoch, loader, n_samples=2):
        """Guarda figuras de campos de vorticidad real y generado."""
        self.G.eval()
        # Tomar algunos ejemplos del val_loader
        seq_in, seq_out, real_traj = next(iter(loader))
        seq_in = seq_in.to(self.device)
        seq_out = seq_out.to(self.device)
        fake_traj, _ = self._generate_sequence(seq_in)
        # Seleccionar primeros n_samples
        seq_in = seq_in[:n_samples].cpu()
        real_traj = real_traj[:n_samples].cpu()
        fake_traj = fake_traj[:n_samples].cpu()
        seq_len = seq_in.size(1)
        fig, axes = plt.subplots(n_samples, seq_len+1, figsize=(4*(seq_len+1), 4*n_samples))
        for i in range(n_samples):
            for t in range(seq_len+1):
                ax = axes[i, t] if n_samples > 1 else axes[t]
                ax.imshow(real_traj[i, t], cmap='RdBu_r', vmin=-2, vmax=2)
                ax.set_title(f"Real t={t}" if i==0 else "")
                ax.axis('off')
        plt.suptitle(f"Época {epoch} - Campos reales")
        plt.tight_layout()
        plt.savefig(self.log_dir / f"real_epoch{epoch:04d}.png")
        plt.close()

        fig, axes = plt.subplots(n_samples, seq_len+1, figsize=(4*(seq_len+1), 4*n_samples))
        for i in range(n_samples):
            for t in range(seq_len+1):
                ax = axes[i, t] if n_samples > 1 else axes[t]
                ax.imshow(fake_traj[i, t], cmap='RdBu_r', vmin=-2, vmax=2)
                ax.set_title(f"Fake t={t}" if i==0 else "")
                ax.axis('off')
        plt.suptitle(f"Época {epoch} - Campos generados")
        plt.tight_layout()
        plt.savefig(self.log_dir / f"fake_epoch{epoch:04d}.png")
        plt.close()
        self.G.train()

    def plot_losses(self):
        """Genera una figura con las curvas de pérdida."""
        fig, axs = plt.subplots(2, 2, figsize=(12, 8))
        axs[0,0].plot(self.history["loss_D"], label='D loss')
        axs[0,0].plot(self.history["loss_G"], label='G loss')
        axs[0,0].legend()
        axs[0,0].set_title('Losses')
        axs[0,1].plot(self.history["w_dist"], label='Wasserstein distance')
        axs[0,1].legend()
        axs[0,1].set_title('W-dist')
        axs[1,0].plot(self.history["loss_mse"], label='MSE')
        axs[1,0].plot(self.history["loss_phys"], label='Physics')
        axs[1,0].legend()
        axs[1,0].set_title('Generator terms')
        axs[1,1].plot(self.history["val_mse"], label='Val MSE')
        axs[1,1].legend()
        axs[1,1].set_title('Validation')
        plt.tight_layout()
        plt.savefig(self.log_dir / "losses.png")
        plt.close()

    def load_checkpoint(self, path):
        ckpt = torch.load(path, map_location=self.device)
        self.G.load_state_dict(ckpt["G"])
        self.D.load_state_dict(ckpt["D"])
        self.opt_G.load_state_dict(ckpt["opt_G"])
        self.opt_D.load_state_dict(ckpt["opt_D"])
        if self.use_scheduler and "sched_G" in ckpt and "sched_D" in ckpt:
            self.sched_G.load_state_dict(ckpt["sched_G"])
            self.sched_D.load_state_dict(ckpt["sched_D"])
        logger.info(f"Checkpoint cargado — época {ckpt['epoch']}, val MSE {ckpt['val_mse']:.6f}")
        return ckpt["epoch"]

    def save_checkpoint(self, epoch, val_mse):
        torch.save({
            "epoch": epoch,
            "G": self.G.state_dict(),
            "D": self.D.state_dict(),
            "opt_G": self.opt_G.state_dict(),
            "opt_D": self.opt_D.state_dict(),
            "sched_G": self.sched_G.state_dict() if self.use_scheduler else None,
            "sched_D": self.sched_D.state_dict() if self.use_scheduler else None,
            "val_mse": val_mse,
        }, self.log_dir / "best_checkpoint_wgafnogp.pt")

    def fit(self, train_loader, val_loader, n_epochs, log_every=10):
        best_val = float("inf")
        self._critic_iter = iter(train_loader)

        # Tomar un batch de validación para visualizaciones fijas
        if self.val_example is None:
            self.val_example = next(iter(val_loader))

        for epoch in range(1, n_epochs + 1):
            self.G.train()
            self.D.train()
            ep = {"ld": [], "wd": [], "lg": [], "mse": [], "phys": []}

            # Barra de progreso por época
            pbar = tqdm(train_loader, desc=f"Epoch {epoch:4d}/{n_epochs}", unit="batch")
            for seq_in, seq_out, _ in pbar:
                seq_in = seq_in.to(self.device)
                seq_out = seq_out.to(self.device)

                ld, wd = self._step_D(train_loader)
                lg, lmse, lp = self._step_G(seq_in, seq_out)

                ep["ld"].append(ld)
                ep["wd"].append(wd)
                ep["lg"].append(lg)
                ep["mse"].append(lmse)
                ep["phys"].append(lp)

                # Actualizar barra con métricas
                pbar.set_postfix({
                    "L_D": f"{ld:.3f}",
                    "W-dist": f"{wd:.3f}",
                    "L_G": f"{lg:.3f}",
                    "MSE": f"{lmse:.3f}"
                })

            val_mse, val_energy, val_enstrophy = self._validate(val_loader)

            # Guardar en historial
            self.history["loss_D"].append(np.mean(ep["ld"]))
            self.history["loss_G"].append(np.mean(ep["lg"]))
            self.history["w_dist"].append(np.mean(ep["wd"]))
            self.history["loss_mse"].append(np.mean(ep["mse"]))
            self.history["loss_phys"].append(np.mean(ep["phys"]))
            self.history["val_mse"].append(val_mse)
            self.history["lr_G"].append(self.opt_G.param_groups[0]['lr'])
            self.history["lr_D"].append(self.opt_D.param_groups[0]['lr'])

            # Scheduler step (usando val_mse)
            if self.use_scheduler:
                self.sched_G.step(val_mse)
                self.sched_D.step(val_mse)

            # Checkpoint y visualización
            if val_mse < best_val:
                best_val = val_mse
                self.save_checkpoint(epoch, val_mse)
                logger.info(f"✅ Nuevo mejor modelo guardado (epoch {epoch}, val MSE={val_mse:.6f})")

            if epoch % self.vis_freq == 0:
                self.visualize(epoch, val_loader, n_samples=2)
                self.plot_losses()
                logger.info(f"📊 Visualización guardada en {self.log_dir}")

            if epoch % log_every == 0:
                logger.info(
                    f"Epoch {epoch:4d} | "
                    f"W-dist: {np.mean(ep['wd']):+.4f} | "
                    f"L_D: {np.mean(ep['ld']):+.5f} | "
                    f"L_G: {np.mean(ep['lg']):+.5f} | "
                    f"MSE: {np.mean(ep['mse']):.5f} | "
                    f"Phys: {np.mean(ep['phys']):.4f} | "
                    f"Val MSE: {val_mse:.5f} | "
                    f"Energía: {val_energy:.4f} | "
                    f"Enstrofía: {val_enstrophy:.4f}"
                )

        logger.info(f"\nMejor val MSE: {best_val:.6f}  — guardado en {self.log_dir}/best_checkpoint_wgafnogp.pt")
        return self.history


# ═══════════════════════════════════════════════════════════
# 10. EJEMPLO DE USO
# ═══════════════════════════════════════════════════════════
if __name__ == "__main__":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    DATA_PATH = "../datasets/Snapshots/snapshots_64x64_use.npy"
    H, W = 64, 64
    SEQ_LEN = 5
    EPOCHS = 2

    # Hiperparámetros
    BATCH = 8
    MODES = 8
    HIDDEN = 32
    N_LAYERS = 3
    N_CRITIC = 3
    Z_DIM = 4
    NU = 1e-3
    DT = 0.1
    LAMBDA_GP = 10.0
    ALPHA_MSE = 1.0
    BETA_PHYS = 0.1

    # Dataset
    dataset = KolmogorovDataset(DATA_PATH, seq_len=SEQ_LEN)
    train_len = int(0.8 * len(dataset))
    val_len = len(dataset) - train_len
    train_dataset, val_dataset = random_split(dataset, [train_len, val_len])

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH, shuffle=True,
        num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH, shuffle=False, num_workers=4,
    )

    # Modelos
    G = FNOGenerator(
        hidden_ch=HIDDEN, modes1=MODES, modes2=MODES,
        n_layers=N_LAYERS, z_dim=Z_DIM,
    )
    D = FNODiscriminator(seq_len=SEQ_LEN+1, hidden_ch=HIDDEN, modes1=MODES, modes2=MODES, n_layers=N_LAYERS)
    ns_loss = NavierStokesLoss(H, W, nu=NU, dt=DT, device=DEVICE)

    # Trainer con nuevas opciones
    trainer = WGAFNOGPTrainer(
        generator=G,
        discriminator=D,
        ns_loss=ns_loss,
        device=DEVICE,
        lr_G=1e-4,
        lr_D=1e-4,
        n_critic=N_CRITIC,
        lambda_gp=LAMBDA_GP,
        alpha_mse=ALPHA_MSE,
        beta_phys=BETA_PHYS,
        use_scheduler=True,          # Activar scheduler
        scheduler_patience=3,
        scheduler_factor=0.5,
        log_dir="training_logs",     # Directorio para logs y figuras
        vis_freq=5,                  # Visualizar cada 5 épocas
    )

    history = trainer.fit(train_loader, val_loader, n_epochs=EPOCHS, log_every=1)

    # Rollout final
    sample = next(iter(val_loader))
    w0 = sample[0][:, 0]
    w_true = sample[2]
    roller = Rollout(G, DEVICE)
    traj = roller.run(w0, n_steps=SEQ_LEN, w_true=w_true)
    logger.info("Rollout completado. Métricas:")
    logger.info(f"Energía: {roller.metrics['energy']}")
    logger.info(f"RMSE: {roller.metrics['rmse']}")
    logger.info(f"Error relativo: {roller.metrics['rel_error']}")

2026-03-25 20:07:03,892 | INFO | Dataset cargado: 1152 trayectorias × 95 ventanas = 109,440 muestras  |  seq_len=5  |  H×W=64×64
Epoch    1/2:   0%|                                | 0/10944 [00:00<?, ?batch/s]/home/joseluis/miniconda3/envs/cfd-project-personal/lib/python3.10/site-packages/torch/autograd/graph.py:824: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:181.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
Epoch    1/2:  25%|▎| 2736/10944 [22:57<1:08:52,  1.99batch/s, L_D=-7.940, W-dis
2026-03-25 20:31:25,218 | INFO | ✅ Nuevo mejor modelo guardado (epoch 1, val MSE=0.219780)
2026-03-25 20:31:25,220 | INFO | Epoch    1 | W-dist: +3.4915 | L_D: -2.41500 | L_G: +9.95252 | MSE: 0.20633 | Phys: 17.2973 | Val MSE: 0.21978 | Energía: 0.4937 | Enstrofía: 102972.5039
Epoch    2/2:  25%|

In [12]:
# Crear un campo inicial con espectro de Kolmogorov (ejemplo simplificado)
kx = torch.fft.fftfreq(W, d=1.0).to(DEVICE)
ky = torch.fft.fftfreq(H, d=1.0).to(DEVICE)
KX, KY = torch.meshgrid(kx, ky, indexing='ij')
K = torch.sqrt(KX**2 + KY**2)
# Espectro de energía tipo Kolmogorov: E(k) ~ k^{-5/3} (2D)
# generamos amplitudes con fase aleatoria
amp = K**(-5/6)
amp[0,0] = 0  # media cero
phase = 2*np.pi*torch.rand(H, W, device=DEVICE)
wf = amp * torch.exp(1j * phase)
w0 = torch.fft.ifft2(wf).real.unsqueeze(0).unsqueeze(0)  # (1,1,H,W)

In [15]:
    # Rollout final
sample = next(iter(val_loader))
#w0 = sample[0][:, 0]
w_true = sample[2]
roller = Rollout(G, DEVICE)
traj = roller.run(w0, n_steps=500, w_true=w_true)
logger.info("Rollout completado. Métricas:")
logger.info(f"Energía: {roller.metrics['energy']}")
logger.info(f"RMSE: {roller.metrics['rmse']}")
logger.info(f"Error relativo: {roller.metrics['rel_error']}")

2026-03-25 21:48:21,236 | INFO | Rollout completado. Métricas:
2026-03-25 21:48:21,241 | INFO | Energía: [6.53521973e-04 4.74727079e-02 5.06555513e-02 5.35617732e-02
 6.31314069e-02 6.83640763e-02 7.57220089e-02 8.30088854e-02
 9.17528272e-02 1.02102444e-01 1.13985620e-01 1.23622924e-01
 1.33248791e-01 1.46482676e-01 1.56636223e-01 1.63734823e-01
 1.67986035e-01 1.77083507e-01 1.86688438e-01 1.92463160e-01
 2.01187491e-01 2.08774999e-01 2.17522964e-01 2.22600743e-01
 2.29311168e-01 2.35270694e-01 2.36378193e-01 2.41161719e-01
 2.42562845e-01 2.46579260e-01 2.54910469e-01 2.65832543e-01
 2.82591909e-01 2.95771241e-01 3.03771734e-01 3.09853554e-01
 3.22100163e-01 3.39763433e-01 3.59592140e-01 3.71597648e-01
 3.82962912e-01 3.89469594e-01 4.03528929e-01 4.18632954e-01
 4.36133474e-01 4.50419694e-01 4.65150565e-01 4.75584567e-01
 4.80501175e-01 4.88553196e-01 5.03684878e-01 5.15214622e-01
 5.24341643e-01 5.34677505e-01 5.48096478e-01 5.65877259e-01
 5.89651585e-01 6.10653043e-01 6.28784597

In [16]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# Selecciona el primer ejemplo del lote
seq = traj[0].cpu().numpy()  # (T, H, W)

# Configura la figura
fig, ax = plt.subplots(figsize=(4, 4))
vmin, vmax = -2, 2  # Ajusta según el rango de tus datos
im = ax.imshow(seq[0], cmap='RdBu_r', vmin=vmin, vmax=vmax)
ax.axis('off')

def animate(t):
    im.set_data(seq[t])
    ax.set_title(f't={t}')
    return [im]

ani = animation.FuncAnimation(fig, animate, frames=seq.shape[0], interval=100, blit=True)
ani.save('rollout_3.gif', writer='pillow')  # Necesita pillow instalado
plt.close(fig)

2026-03-25 21:48:21,989 | INFO | Animation.save using <class 'matplotlib.animation.PillowWriter'>


In [5]:
pip install imageio


Note: you may need to restart the kernel to use updated packages.
